<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-09-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-100.

Características do dataset:

- 60.000 imagens RGB
- 100 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar100

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-100 utilizando `tensorflow.keras.datasets.cifar100.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [1]:
from tensorflow.keras.datasets import cifar100
from sklearn.model_selection import train_test_split
import numpy as np


def load_data(seed):

    # Carrega o dataset
    (X_train, y_train), (X_test, y_test) = cifar100.load_data()

    # Junta treino + teste
    X = np.concatenate((X_train, X_test), axis=0)
    y = np.concatenate((y_train, y_test), axis=0)

    # Flatten das imagens
    X = X.reshape(X.shape[0], -1)

    # Normalização
    X = X.astype("float32") / 255.0

    # Separação treino e validação
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_val, y_train, y_val


# Teste
X_train, X_val, y_train, y_val = load_data(seed=42)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step
X_train: (48000, 3072)
X_val: (12000, 3072)
y_train: (48000, 1)
y_val: (12000, 1)


1. Qual o formato original das imagens?

As imagens possuem formato:

(32, 32, 3)

Onde:

32 → altura;
32 → largura;
3 → canais RGB (vermelho, verde e azul).

2. Quantas features cada imagem possui após o flatten?

Após o flatten, cada imagem possui:

32 × 32 × 3 = 3072

features.

32×32×3=3072

Logo, cada imagem é transformada em um vetor de 3072 posições.

3. Por que o flatten é necessário para uma MLP?

A MLP (Multilayer Perceptron) trabalha com vetores unidimensionais como entrada. As imagens originalmente possuem estrutura tridimensional (altura, largura, canais), então é necessário converter cada imagem em um vetor linear para que a rede consiga processar os dados corretamente.

4. Qual a importância da normalização para o treinamento?

A normalização reduz os valores dos pixels para o intervalo entre 0 e 1, dividindo por 255.

Isso é importante porque:

melhora a estabilidade do treinamento;
acelera a convergência;
evita valores muito altos nas entradas;
ajuda o algoritmo de otimização a aprender melhor.

Sem normalização, o treinamento pode ficar mais lento e menos eficiente.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [9]:
from sklearn.neural_network import MLPClassifier


def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):

    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=20,
        batch_size=256,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        verbose=True
    )

    model.fit(X_train, y_train.ravel())

    return model

In [10]:
model = train_mlp(
    X_train=X_train,
    y_train=y_train,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    seed=42
)

print(model)

Iteration 1, loss = 4.55691471
Validation score: 0.021042
Iteration 2, loss = 4.34875591
Validation score: 0.034792
Iteration 3, loss = 4.22051379
Validation score: 0.050208
Iteration 4, loss = 4.16531258
Validation score: 0.054375
Iteration 5, loss = 4.14091427
Validation score: 0.054583
Iteration 6, loss = 4.12677612
Validation score: 0.058333
Iteration 7, loss = 4.11387558
Validation score: 0.059167
Iteration 8, loss = 4.10236504
Validation score: 0.061042
Iteration 9, loss = 4.09371708
Validation score: 0.062083
Iteration 10, loss = 4.08628841
Validation score: 0.073125
Iteration 11, loss = 4.01978703
Validation score: 0.079375
Iteration 12, loss = 3.98071614
Validation score: 0.085417
Iteration 13, loss = 3.95399824
Validation score: 0.094375
Iteration 14, loss = 3.93861335
Validation score: 0.100417
Iteration 15, loss = 3.92443248
Validation score: 0.098125
Iteration 16, loss = 3.91993026
Validation score: 0.094792
Iteration 17, loss = 3.91344553
Validation score: 0.100833
Iterat

c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


1. Quantos parâmetros existem na primeira camada?

No seu modelo, a primeira camada possui:

hidden_layer_sizes=(64,)

Ou seja:

entrada = 3072 features;
64 neurônios na camada oculta.

Cada neurônio possui:

3072 pesos;
1 bias.

Então:

(3072+1)×64=196672

Logo, a primeira camada possui:

196672 parâmetros

2. Qual a função da ativação ReLU?

A ReLU (Rectified Linear Unit) é uma função de ativação que transforma valores negativos em zero e mantém valores positivos.

f(x)=max(0,x)

Ela é importante porque:

acelera o treinamento;
reduz o problema do gradiente desaparecer;
aumenta a eficiência computacional;
ajuda a rede a aprender padrões complexos.

3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

Porque imagens possuem muitas features.

No CIFAR-100:

32×32×3=3072

Cada pixel da imagem vira uma entrada da rede.

Como cada neurônio da MLP conecta-se a todas as entradas da camada anterior, o número de pesos cresce muito rapidamente, aumentando:

custo computacional;
uso de memória;
tempo de treinamento;
risco de overfitting.

Por isso CNNs geralmente são mais eficientes para imagens do que MLPs.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd


def evaluate(model, X_test, y_test):

    # Predições
    y_pred = model.predict(X_test)

    # Métricas
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    # Resultados
    results = pd.DataFrame({
        "Métrica": ["Accuracy", "Precision", "Recall", "F1-Score"],
        "Valor": [accuracy, precision, recall, f1]
    })

    return results

In [14]:
results = evaluate(model, X_val, y_val)

print(results)

     Métrica     Valor
0   Accuracy  0.097500
1  Precision  0.071056
2     Recall  0.097500
3   F1-Score  0.068709


Os resultados mostram que a MLP conseguiu aprender alguns padrões do dataset CIFAR-100, porém o desempenho ainda foi relativamente baixo. Isso acontece porque o CIFAR-100 é um problema complexo, contendo 100 classes diferentes e imagens pequenas, o que dificulta a classificação utilizando uma MLP simples.

A accuracy de aproximadamente 9,75% indica que o modelo acertou cerca de 10 imagens a cada 100 avaliadas. Apesar de parecer baixa, esse resultado já é superior ao acaso puro, que seria próximo de 1% em um problema com 100 classes.

Accuracy=0.0975≈9.75%

A precision de aproximadamente 7,1% mostra que muitas previsões positivas realizadas pelo modelo ainda não foram corretas, indicando dificuldade em distinguir adequadamente as classes.

Precision=0.071056≈7.1%

O recall de aproximadamente 9,75% indica que o modelo conseguiu identificar apenas uma pequena parcela dos exemplos corretos de cada classe.

Recall=0.0975≈9.75%

O F1-score de aproximadamente 6,87% mostra que o equilíbrio entre precision e recall ainda é baixo, reforçando que o modelo possui dificuldade para generalizar corretamente no CIFAR-100.

F1=0.068709≈6.87%

De forma geral, os resultados evidenciam as limitações das MLPs para tarefas de visão computacional, já que elas não conseguem explorar características espaciais das imagens da mesma forma que redes convolucionais (CNNs).

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [15]:
import mlflow
import time
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    max_iter,
    batch_size,
    seed
):

    with mlflow.start_run():

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            random_state=seed,
            max_iter=max_iter,
            batch_size=batch_size,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=5
        )

        start_time = time.time()

        model.fit(X_train, y_train.ravel())

        training_time = time.time() - start_time

        y_pred = model.predict(X_val)

        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, average="macro", zero_division=0)
        recall = recall_score(y_val, y_pred, average="macro", zero_division=0)
        f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("training_time", training_time)

        results = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "training_time": training_time
        }

        return model, results

In [16]:
experiments = []

configs = [
    ("relu", (64,), 0.001, 20, 256),
    ("relu", (128,), 0.001, 20, 256),
    ("tanh", (64,), 0.001, 20, 256),
    ("relu", (64,), 0.01, 20, 256),
]

for activation, hidden_layers, learning_rate, max_iter, batch_size in configs:
    model, results = run_experiment(
        X_train,
        y_train,
        X_val,
        y_val,
        activation,
        hidden_layers,
        learning_rate,
        max_iter,
        batch_size,
        seed=42
    )

    experiments.append(results)

df_results = pd.DataFrame(experiments)
df_results

2026/05/26 21:42:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/26 21:42:16 INFO mlflow.store.db.utils: Updating database tables
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,training_time
0,relu,"(64,)",0.001,20,256,0.097500,0.071056,0.097500,0.068709,43.760926
1,relu,"(128,)",0.001,20,256,0.162333,0.139572,0.162333,0.135935,77.514904
2,tanh,"(64,)",0.001,20,256,0.176250,0.162749,0.176250,0.157065,32.909879
3,relu,"(64,)",0.010,20,256,0.010000,0.000100,0.010000,0.000198,18.113535


1. Qual experimento apresentou melhor desempenho?

O melhor experimento deve ser escolhido pelo maior f1_score e pela maior accuracy. No caso do CIFAR-100 com MLP, normalmente a configuração com activation="relu", hidden_layers=(64,), learning_rate=0.001, max_iter=20 e batch_size=256 tende a apresentar desempenho mais estável e adequado para comparação inicial.

2. Qual configuração apresentou maior estabilidade?

A configuração mais estável foi a que utilizou learning_rate=0.001, pois ela apresenta aprendizado mais gradual. Taxas maiores, como 0.01, podem causar oscilações maiores na loss e dificultar a convergência.

3. Qual o benefício do rastreamento experimental?

O MLflow permite registrar parâmetros, métricas e tempo de treinamento de cada experimento. Com isso, fica mais fácil comparar configurações diferentes, identificar qual modelo teve melhor desempenho e justificar as escolhas feitas durante o desenvolvimento.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [17]:
activations = ["logistic", "tanh", "relu"]

results_q5 = []

for activation in activations:
    model, results = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
        max_iter=20,
        batch_size=256,
        seed=42
    )

    results_q5.append(results)

df_q5 = pd.DataFrame(results_q5)
df_q5

c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,training_time
0,logistic,"(64,)",0.001,20,256,0.186417,0.178002,0.186417,0.166185,36.816910
1,tanh,"(64,)",0.001,20,256,0.176250,0.162749,0.176250,0.157065,33.081679
2,relu,"(64,)",0.001,20,256,0.097500,0.071056,0.097500,0.068709,32.745865


1. Qual ativação apresentou melhor convergência?

A função logistic apresentou melhor convergência neste experimento, obtendo os maiores valores de accuracy e f1-score. Isso indica que ela conseguiu aprender melhor os padrões do dataset dentro do número de iterações utilizado.

Accuracy
logistic
	​

=0.186417

2. Qual ativação apresentou maior estabilidade?

A ativação logistic também apresentou maior estabilidade, pois manteve desempenho superior nas métricas avaliadas em comparação com tanh e relu.

3. Houve diferenças significativas no treinamento?

Sim. Os resultados mostraram diferenças relevantes entre as funções de ativação. A logistic apresentou melhor desempenho geral, seguida pela tanh. Já a relu apresentou resultados inferiores neste experimento específico.

Isso pode ocorrer devido:

à arquitetura utilizada;
ao número reduzido de iterações;
à configuração dos hiperparâmetros;
às características do CIFAR-100.

4. Por que a ReLU é amplamente utilizada em Deep Learning?

A ReLU é amplamente utilizada porque possui baixo custo computacional e ajuda a reduzir o problema do desaparecimento do gradiente. Além disso, normalmente acelera o treinamento em redes profundas.

f(x)=max(0,x)

Mesmo que neste experimento ela não tenha obtido o melhor resultado, em arquiteturas mais profundas e em muitos problemas reais a ReLU costuma apresentar excelente desempenho.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [19]:
architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

results_q6 = []

for hidden_layers in architectures:
    model, results = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="logistic",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        max_iter=20,
        batch_size=256,
        seed=42
    )

    results_q6.append(results)

df_q6 = pd.DataFrame(results_q6)
df_q6

c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,training_time
0,logistic,"(32,)",0.001,20,256,0.161250,0.137827,0.161250,0.128932,26.262165
1,logistic,"(64,)",0.001,20,256,0.186417,0.178002,0.186417,0.166185,35.455103
2,logistic,"(128, 64)",0.001,20,256,0.168417,0.148219,0.168417,0.142098,109.276239
3,logistic,"(256, 128)",0.001,20,256,0.198583,0.186884,0.198583,0.177208,222.571177


1. Redes maiores sempre melhoraram os resultados?

Não totalmente. Houve melhora de desempenho conforme a arquitetura aumentou, porém o ganho não foi proporcional ao aumento do custo computacional.

Por exemplo:

(32,) → accuracy ≈ 16,1%
(64,) → accuracy ≈ 18,6%
(128, 64) → accuracy ≈ 16,8%
(256, 128) → accuracy ≈ 19,8%

Accuracy
(256,128)
	​

=0.198583

A arquitetura (128, 64) inclusive teve desempenho inferior à (64,), mostrando que aumentar a rede nem sempre melhora os resultados.

2. Qual arquitetura apresentou melhor tradeoff?

A arquitetura (64,) apresentou o melhor tradeoff entre desempenho e custo computacional.

Ela obteve:

accuracy relativamente alta (~18,6%);
bom f1-score (~16,6%);
tempo de treinamento muito menor que (256, 128).

Enquanto (256, 128) teve melhor accuracy, o custo computacional aumentou bastante.

3. Houve sinais de overfitting?

Houve indícios leves de overfitting nas arquiteturas maiores.

A arquitetura (256, 128) aumentou significativamente o tempo de treinamento, mas o ganho nas métricas foi relativamente pequeno quando comparado à (64,).

Isso sugere que o aumento da complexidade da rede não trouxe melhoria proporcional na generalização.

4. Qual arquitetura apresentou maior custo computacional?

A arquitetura (256, 128) apresentou o maior custo computacional.

Ela teve o maior tempo de treinamento:

training_time
(256,128)
	​

≈222.57s

Isso ocorreu porque redes maiores possuem:

mais neurônios;
mais pesos;
mais operações matemáticas durante o treinamento.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [20]:
learning_rates = [0.1, 0.01, 0.001]

results_q7 = []

for lr in learning_rates:
    model, results = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="logistic",
        hidden_layers=(64,),
        learning_rate=lr,
        max_iter=20,
        batch_size=256,
        seed=42
    )

    results_q7.append(results)

df_q7 = pd.DataFrame(results_q7)
df_q7

c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,training_time
0,logistic,"(64,)",0.100,20,256,0.010000,0.000100,0.010000,0.000199,60.615077
1,logistic,"(64,)",0.010,20,256,0.082667,0.047206,0.082667,0.047215,73.614831
2,logistic,"(64,)",0.001,20,256,0.186417,0.178002,0.186417,0.166185,60.259641


1. Qual learning rate apresentou melhor desempenho?

O learning rate 0.001 apresentou o melhor desempenho no experimento, obtendo os maiores valores de accuracy, precision, recall e f1-score.

Accuracy
0.001
	​

=0.186417

Além disso, ele apresentou treinamento mais estável e melhor convergência.

2. Qual apresentou maior instabilidade?

O learning rate 0.1 apresentou maior instabilidade.

Ele obteve desempenho muito baixo:

accuracy ≈ 1%;
f1-score praticamente zero.

Accuracy
0.1
	​

=0.01

Isso indica que o modelo não conseguiu convergir adequadamente durante o treinamento.

3. O que acontece quando o learning rate é muito alto?

Quando o learning rate é muito alto, o modelo realiza atualizações muito grandes nos pesos, causando instabilidade no treinamento.

Isso pode fazer com que:

a loss oscile;
o modelo “pule” o ponto ideal;
a convergência não aconteça corretamente.

No experimento, isso ocorreu com learning_rate = 0.1.

4. O que acontece quando o learning rate é muito baixo?

Quando o learning rate é muito baixo, o treinamento ocorre de forma mais lenta, pois os pesos são atualizados com passos pequenos.

Apesar disso, taxas menores costumam gerar treinamento mais estável e melhor convergência, como ocorreu com learning_rate = 0.001.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

Durante os experimentos realizados com o dataset CIFAR-100, foi possível observar diferenças importantes no comportamento do treinamento da MLP de acordo com a função de ativação, arquitetura da rede e learning rate utilizados.

A loss apresentou redução gradual ao longo das iterações, indicando que o modelo conseguiu aprender padrões presentes nos dados. Entretanto, em algumas configurações o treinamento não convergiu completamente dentro do limite de iterações definido, gerando warnings de convergência. Isso ocorreu principalmente devido à complexidade do CIFAR-100 e à grande quantidade de parâmetros presentes na MLP.

O learning rate teve impacto direto na estabilidade e no desempenho do treinamento. O valor 0.001 apresentou melhor convergência e melhores métricas, enquanto 0.1 causou grande instabilidade, resultando em desempenho muito baixo.

Accuracy
0.001
	​

=0.186417

Arquiteturas maiores aumentaram a capacidade de aprendizado da rede, porém também elevaram significativamente o custo computacional. A arquitetura (256, 128) apresentou melhor accuracy, mas o tempo de treinamento aumentou bastante.

training_time
(256,128)
	​

≈222.57s

Em relação às funções de ativação, a logistic apresentou melhor desempenho nos experimentos realizados, seguida pela tanh. A relu, apesar de amplamente utilizada em Deep Learning, obteve desempenho inferior neste cenário específico.

O treinamento da MLP mostrou-se computacionalmente custoso, especialmente para arquiteturas maiores, pois imagens possuem muitas features após o flatten. Como cada neurônio conecta-se totalmente à camada anterior, o número de parâmetros cresce rapidamente.

32×32×3=3072

Além disso, as MLPs possuem limitações para processamento de imagens, pois não conseguem explorar adequadamente informações espaciais e padrões locais presentes nos pixels, diferentemente das CNNs.

O backpropagation foi essencial para o aprendizado da rede, pois permitiu calcular os gradientes do erro e atualizar os pesos da MLP iterativamente, reduzindo a loss ao longo do treinamento.

1. Qual configuração apresentou melhor resultado final?

A configuração que apresentou melhor resultado final foi:

ativação logistic;
arquitetura (256, 128);
learning rate 0.001.

Essa configuração obteve a maior accuracy e o maior f1-score entre os experimentos realizados.

2. Quais foram as principais dificuldades observadas?

As principais dificuldades observadas foram:

alto custo computacional;
tempo elevado de treinamento;
warnings de convergência;
dificuldade da MLP em lidar com imagens complexas;
baixo desempenho em algumas configurações de learning rate e ativação.

3. Por que MLPs possuem limitações para imagens?

Porque as MLPs tratam os pixels apenas como vetores lineares após o flatten, perdendo informações espaciais importantes da imagem.

Isso dificulta a identificação de:

bordas;
texturas;
padrões locais;
relações espaciais entre pixels.

Por isso CNNs costumam apresentar desempenho muito superior em tarefas de visão computacional.

4. Como o backpropagation contribui para o aprendizado da rede?

O backpropagation calcula o erro da rede e propaga esse erro de volta pelas camadas da MLP.

Com isso, os pesos são ajustados gradualmente utilizando gradientes calculados a partir da função de perda.

Esse processo permite que a rede aprenda padrões nos dados e reduza a loss ao longo do treinamento.